In [1]:
import os
from pathlib import Path
import numpy as np

import pandas as pd

In [2]:
project_root = Path(os.environ["PROJECT_ROOT"])

In [3]:
hcp_pheno = pd.read_csv(os.environ["HCP_PHENO_UNRESTRICTED"], dtype={"Subject": str})
hcp_pheno.set_index("Subject", inplace=True)
print(hcp_pheno.iloc[:, :4].head(5))

        Release Acquisition Gender    Age
Subject                                  
100004     S900         Q06      M  22-25
100206     S900         Q11      M  26-30
100307       Q1         Q01      F  26-30
100408       Q3         Q03      M  31-35
100610     S900         Q08      M  26-30


In [4]:
hcp_58_columns = (
    (project_root / "resources/column_lists/58behaviors.txt").read_text().splitlines()
)

hcp_behav = hcp_pheno.loc[:, hcp_58_columns]

complete_behav_mask = ~hcp_behav.isna().any(axis=1)
print("Num with complete behavioral data:", complete_behav_mask.sum())

Num with complete behavioral data: 1022


In [5]:
complete_rest_mask = hcp_pheno["3T_RS-fMRI_Count"] == 4
print("Num with complete 3T rest data:", complete_rest_mask.sum())

Num with complete 3T rest data: 1018


In [6]:
print(
    "Num with complete behavioral and 3T rest data:",
    (complete_behav_mask & complete_rest_mask).sum(),
)

Num with complete behavioral and 3T rest data: 987


In [7]:
hcp_fd_dir = project_root / "results/hcp_1200_rfmri_fd"
hcp_fd = pd.read_parquet(hcp_fd_dir / "hcp_1200_rfmri_fd.parquet")

# Only include 3T data and full runs.
hcp_fd = hcp_fd.query("mag == '3T' and n_frames == 1200")

# Eclude runs with more than 50% censored volumes
censor_threshold = 0.5
hcp_fd = hcp_fd.query(f"censor_frac < {censor_threshold}")

# Aggregate over subject
hcp_fd = hcp_fd.groupby("sub").agg({"task": "count", "mean_fd": "mean"})
hcp_fd.columns = ["run_count", "mean_fd"]
hcp_fd.index.name = "Subject"

print(hcp_fd.head(5))

         run_count   mean_fd
Subject                     
100206           4  0.108884
100307           4  0.125204
100408           4  0.183484
100610           4  0.180003
101006           4  0.155110


In [7]:
censor_complete_rest_mask = hcp_fd["run_count"] == 4
print(
    "Num with complete 3T rest data (after censoring):", censor_complete_rest_mask.sum()
)

Num with complete 3T rest data (after censoring): 893


In [8]:
mean_fd_threshold = 0.3
low_fd_mask = hcp_fd["mean_fd"] < mean_fd_threshold
print(f"Num with mean FD < {mean_fd_threshold}:", low_fd_mask.sum())

Num with mean FD < 0.3: 1063


In [9]:
hcp_include_mask = complete_behav_mask & censor_complete_rest_mask & low_fd_mask
print("Total num include:", hcp_include_mask.sum())

Total num include: 867


In [10]:
print("\nbehav:\n", complete_behav_mask[:10])
print("\ncomplete:\n", censor_complete_rest_mask[:10])
print("\nfd:\n", low_fd_mask[:10])
print("\nfinal:\n", hcp_include_mask[:10])


behav:
 Subject
100004    False
100206     True
100307     True
100408     True
100610     True
101006     True
101107     True
101208    False
101309     True
101410     True
dtype: bool

complete:
 Subject
100206     True
100307     True
100408     True
100610     True
101006     True
101107     True
101309     True
101410    False
101915     True
102008    False
Name: run_count, dtype: bool

fd:
 Subject
100206    True
100307    True
100408    True
100610    True
101006    True
101107    True
101309    True
101410    True
101915    True
102008    True
Name: mean_fd, dtype: bool

final:
 Subject
100004    False
100206     True
100307     True
100408     True
100610     True
101006     True
101107     True
101208    False
101309     True
101410    False
dtype: bool


N = 867 subjects is comparable with the N = 835 subjects used in [Kong et al., 2023](https://doi.org/10.1016/j.neuroimage.2023.120044).

In [11]:
rng = np.random.default_rng(7582)
hcp_include_subs = hcp_include_mask.index[hcp_include_mask].values
# Shuffle order so that each subsequence is a random sample
rng.shuffle(hcp_include_subs)
print(hcp_include_subs[:8])

['181131' '245333' '106824' '187547' '211821' '158136' '139233' '744553']


In [12]:
n_subs = len(hcp_include_subs)

subject_list_path = (
    project_root
    / "resources"
    / "subject_lists"
    / f"hcp_complete_data_{n_subs}_subject_list.txt"
)
np.savetxt(subject_list_path, hcp_include_subs, fmt="%s")